<a href="https://colab.research.google.com/github/sadbinsiddique/Dl-net/blob/main/notebook/3.%20Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Training with **Colab**

<div style="display: flex; flex-direction: column; align-items: center; justify-content: center; padding: 20px;">
    <div style="display: flex; gap: 12px; align-items: center; justify-content: center; flex-wrap: wrap;">
        <a href="https://colab.research.google.com/github/sadbinsiddique/Dl-net/blob/main/notebook/3.%20Train.ipynb" target="_blank" style="display: inline-flex; align-items: center; justify-content: center; gap: 6px; padding: 8px 16px; background: rgba(255, 255, 255, 0.15); backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px); color: white; border-radius: 12px; text-decoration: none; font-weight: 600; font-size: 13px; border: 1px solid rgba(255, 255, 255, 0.25); cursor: pointer; transition: all 0.3s ease;">
            <img src="https://github.com/sadbinsiddique/Dl-net/blob/main/public/google-colab.svg?raw=1" alt="Online" style="width: 20px; height: 20px;">
            Google Colab
        </a>
        <a href="https://github.com/settings/tokens" target="_blank" style="display: inline-flex; align-items: center; justify-content: center; gap: 6px; padding: 8px 16px; background: rgba(255, 255, 255, 0.15); backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px); color: white; border-radius: 12px; text-decoration: none; font-weight: 600; font-size: 13px; border: 1px solid rgba(255, 255, 255, 0.25); cursor: pointer; transition: all 0.3s ease;">
            <img src="https://github.com/sadbinsiddique/Dl-net/blob/main/public/github.png?raw=1" alt="Active" style="width: 20px; height: 20px;">
            Access Token
        </a>
    </div>
</div>

In [ ]:
!pip install imagehash dotenv accelerate

In [ ]:
import os
import torch
import importlib
import pandas as pd
import torch.nn as nn
from PIL import Image
from tqdm import tqdm
import torch.optim as optim
from getpass import getpass
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Cloud-Base Environment: {IN_COLAB}")

if IN_COLAB:
    env="colab"
    DATA_PATH = '/content/Dl-net/data/'
    token = getpass('Enter GitHub Access Token: ')
    !git clone https://{token}@github.com/sadbinsiddique/Dl-net.git
    os.chdir('/content/Dl-net')
else:
    env="local"
    DATA_PATH = './'
    print("Local environment detected. \nSkipping git clone.")

In [ ]:
!sed -i 's/^from eda import \*/from .eda import \*/' /content/Dl-net/src/utils/eda_imp.py

from src.utils import pipeline
importlib.reload(pipeline)
from src.utils.pipeline import _run_pipeline
_run_pipeline()

### Distributed Training Setup (PyTorch DDP)
This section writes our model training logic into a Python script and executes it utilizing PyTorch's `torchrun`. It includes SyncBatchNorm, dynamic Learning Rate scaling, and restricts logging/saving to Rank 0 to prevent corruption.

In [ ]:
%%writefile distributed_train.py
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
import torchvision.transforms as transforms
from PIL import Image

# 1. Setup Distributed Environment
def setup():
    dist.init_process_group("nccl")
    rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(rank)
    return rank

def cleanup():
    dist.destroy_process_group()

# 2. Dataset Integration
# Adjust this class to load your specific pipeline data (e.g., CK+ or CASME II CSV mapping)
class EmotionDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        # Load actual paths from the data_dir. Using a stub here for demonstration.
        self.image_paths = glob.glob(os.path.join(data_dir, "*/*.png")) # Adjust based on your dir structure
        self.transform = transform
        self.labels = [0] * len(self.image_paths) # Replace with actual label extraction logic

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        # Load and convert image to RGB
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

# 3. Main Training Loop
def main():
    rank = setup()
    world_size = int(os.environ["WORLD_SIZE"])
    
    if rank == 0:
        print(f"Running DDP Training on {world_size} GPUs.")
    
    # Hyperparameters
    batch_size = 32
    base_lr = 0.001
    epochs = 10
    lr = base_lr * world_size # Scale learning rate
    num_classes = 7 # Adjust based on CK+/CASME dataset

    # 4. Model definition
    model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(rank)
    
    # Apply SyncBatchNorm and wrap in DDP
    model = nn.SyncBatchNorm.convert_sync_batchnorm(model)
    ddp_model = DDP(model, device_ids=[rank])
    
    # 5. Transforms & DataLoader
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Make sure this points to the exact dir your pipeline outputs
    dataset = EmotionDataset(data_dir='./data', transform=transform)
    
    # Distributed Sampler (Essential for splitting data across GPUs)
    sampler = DistributedSampler(dataset)
    dataloader = DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True)
    
    optimizer = optim.Adam(ddp_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # 6. Execution
    for epoch in range(epochs):
        sampler.set_epoch(epoch) # Ensures data is shuffled differently each epoch
        ddp_model.train()
        
        total_loss = 0.0
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            inputs, targets = inputs.to(rank), targets.to(rank)
            
            optimizer.zero_grad()
            outputs = ddp_model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        # 7. Restrict logging and saving to Rank 0 only
        if rank == 0:
            avg_loss = total_loss / len(dataloader)
            print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f}")
            # Un-wrap from DDP before saving (.module)
            torch.save(ddp_model.module.state_dict(), "best_emotion_model.pth")
            
    cleanup()

if __name__ == "__main__":
    main()


In [ ]:
import torch
num_gpus = torch.cuda.device_count()
print(f"Found {num_gpus} GPUs.")

if num_gpus > 0:
    !torchrun --nproc_per_node={num_gpus} distributed_train.py
else:
    print("No GPUs available. Distributed training requires at least 1 GPU (preferably 2+). Please attach a GPU instance.")